# Customer Feedback Analysis: AI-Powered Workflow
## A hands-on example mirroring real production patterns

# AI-Powered Customer Feedback Analysis: End-to-End Workflow

This notebook walks through a complete, production-style pipeline for analyzing customer complaint data using modern AI tools. If you are starting a role on an AI or data team that works with customer feedback, this workflow reflects the kinds of patterns you will encounter in real projects.

---

## What This Notebook Demonstrates

We will analyze a dataset of synthetic customer complaints stored in a local SQLite database. Rather than just running simple queries or counting words, we will build a multi-stage AI pipeline that classifies, summarizes, and reasons over the data — the same way a production AI team would approach a backlog of thousands of customer messages.

---

## The 5 Main Stages

**Stage 1 — Load & Chunk Data**
Load complaint records from SQLite into pandas, then split them into manageable chunks. Chunking is a foundational pattern in AI pipelines: LLMs have context limits, and real datasets are always larger than a single prompt can hold. We batch our data so every record gets processed.

**Stage 2 — LLM Classification via Anthropic API**
Send each chunk of complaints to Claude (via the Anthropic API) and ask it to classify complaints by theme, urgency, or root cause — going beyond the simple `sentiment_label` already in the data. This is structured output extraction: giving an LLM a schema and asking it to fill it in.

**Stage 3 — LLM Summarization & Synthesis**
After classification, use Claude to write concise summaries across product categories and channels. Summarization at scale (many records → a few key insights) is one of the most common LLM use cases in business intelligence and customer experience teams.

**Stage 4 — LangGraph Agentic Workflow**
Combine the previous steps into an agent built with LangGraph. The agent uses tools (SQL queries, summarization calls) to reason about the data, decide what to look at next, and produce a structured report. This introduces the concept of tool-using LLM agents — a core pattern in production AI systems.

**Stage 5 — Results & Insights**
Review the outputs: classifications, summaries, and agent-generated findings. We will look at what the pipeline surfaced, where it struggled, and how you would extend or productionize it.

---

## Tech Stack

| Tool | Role |
|---|---|
| **SQLite + `sqlite3`** | Local database holding the complaint records |
| **pandas** | Loading, filtering, and reshaping tabular data |
| **Anthropic Claude API** | LLM calls for classification and summarization |
| **LangChain** | Prompt templates, output parsers, and tool wrappers |
| **LangGraph** | Orchestrating multi-step agentic workflows |

---

## Why This Pattern Matters

Production AI teams rarely send one record to one LLM call and call it done. The patterns here — **chunking large datasets**, **structured LLM extraction**, **chained summarization**, and **tool-using agents** — appear constantly in real pipelines for support ticket triage, Voice of Customer analysis, churn prediction, and compliance review.

Working through this notebook will give you hands-on familiarity with each layer of that stack, using a dataset that feels like what you will actually see on the job.

---

*Dataset: synthetic customer complaints across 10 product categories (Credit Card, Mortgage, Personal Loan, and more), with fields for channel, sentiment, resolution status, and free-text complaint body.*

## Stage 1: Setup & Data Loading

In [ ]:
# ── Cell 1: install dependencies (run once, then restart kernel) ──────────────
# !pip install langchain langchain-anthropic langgraph anthropic pandas

# ── Cell 2: imports ────────────────────────────────────────────────────
import sqlite3
import json
import os
import textwrap

import pandas as pd
import anthropic

# ── Cell 3: load complaints from SQLite ──────────────────────────────────

# Path is relative to the notebooks/ directory where this notebook lives
DB_PATH = "../data/complaints.db"

# Open a connection, read the full complaints table into a DataFrame, then close
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM complaints", conn)
conn.close()

# Quick sanity check: how many rows and columns did we load?
print(f"Loaded {df.shape[0]:,} complaints with {df.shape[1]} columns.")
print()

# Preview the first few rows so we know what we are working with
df.head(3)

# ── Cell 4: distributions for key categorical columns ─────────────────────

# product_category tells us which part of the business each complaint touches
print("=== product_category ===")
print(df["product_category"].value_counts())
print()

# sentiment_label is a pre-assigned signal (positive / neutral / negative)
# that we will use later as a ground-truth label for classification experiments
print("=== sentiment_label ===")
print(df["sentiment_label"].value_counts())

## Stage 2: Data Chunking Strategies for LLM Pipelines

In [ ]:
# ============================================================
# DATA CHUNKING FOR LLM PIPELINES
# ============================================================
# Why do we chunk data before sending it to an LLM?
#
# 1. CONTEXT WINDOWS: LLMs can only process a limited amount of
#    text at once (e.g., Claude has a 200k token context window).
#    A large DataFrame of complaint texts could easily exceed this
#    limit, so we must break the data into manageable pieces.
#
# 2. BATCH PROCESSING EFFICIENCY: Processing records in batches
#    lets us pipeline API calls, handle errors per-chunk without
#    losing all work, and resume from a checkpoint if something
#    goes wrong mid-run.
#
# 3. COST CONTROL: LLM APIs charge per token. Chunking lets you
#    process a sample first to validate your prompt and output
#    format before committing to the full dataset — saving money
#    during experimentation.
#
# 4. RATE LIMITS: Most LLM APIs enforce requests-per-minute or
#    tokens-per-minute limits. Chunking naturally paces your calls
#    and makes it easier to add delays or retries between batches.
# ============================================================

import pandas as pd
from typing import Iterator

# --- Strategy 1: Fixed-size row chunks ---
# The simplest and most general approach. Split the DataFrame into
# equal-sized slices regardless of content. Good when each row is
# roughly the same size and you want predictable batch sizes.

def chunk_dataframe(df: pd.DataFrame, chunk_size: int = 10) -> Iterator[pd.DataFrame]:
    """
    Yield successive DataFrame slices of `chunk_size` rows.
    The final chunk may be smaller if len(df) % chunk_size != 0.
    """
    for start in range(0, len(df), chunk_size):
        yield df.iloc[start : start + chunk_size]

# Materialize all chunks into a list so we can inspect them
chunks = list(chunk_dataframe(df, chunk_size=10))

print(f"Total rows in DataFrame : {len(df)}")
print(f"Chunk size requested    : 10 rows")
print(f"Number of chunks created: {len(chunks)}")
print(f"Rows in first chunk     : {len(chunks[0])}")
print(f"Rows in last chunk      : {len(chunks[-1])}  (may be smaller if rows not divisible by chunk size)")

# Quick sanity check — confirm all rows are accounted for
total_rows_across_chunks = sum(len(c) for c in chunks)
assert total_rows_across_chunks == len(df), "Row count mismatch — chunking dropped rows!"
print(f"\nSanity check passed: all {total_rows_across_chunks} rows present across chunks.\n")

# --- Strategy 2: Group by product_category ---
# A semantically meaningful approach. Instead of arbitrary row
# slices, each "chunk" is all complaints for one product category.
# This is realistic when your LLM prompt is category-specific, e.g.:
#   "Summarize the top issues customers report about Credit Cards."
# Grouping ensures the model sees all relevant complaints together,
# which produces more coherent and accurate summaries than mixing
# categories in a single batch.

category_groups = {
    cat: group_df.reset_index(drop=True)          # reset index so each group starts at 0
    for cat, group_df in df.groupby("product_category")
}

print("Category-based chunks (one chunk per product category):")
print(f"{'Category':<35} {'Complaint Count':>15}")
print("-" * 52)
for category, group_df in sorted(category_groups.items()):
    print(f"{category:<35} {len(group_df):>15}")

print(f"\nTotal categories : {len(category_groups)}")
print(f"Total rows check : {sum(len(g) for g in category_groups.values())} (should equal {len(df)})")

# Example: grab just the Credit Card complaints chunk for downstream use
# (replace with whichever category you want to experiment with first)
sample_category = sorted(category_groups.keys())[0]  # first category alphabetically
sample_chunk = category_groups[sample_category]
print(f"\nExample — first category chunk: '{sample_category}'")
print(f"Shape: {sample_chunk.shape}")
print(sample_chunk[["complaint_text"]].head(3))

## Stage 3: LLM Classification (Anthropic API)

In [ ]:
# =============================================================================
# LLM CLASSIFICATION VIA CLAUDE API
#
# This demonstrates a basic "skill" — a reusable agent capability that you can
# call repeatedly to enrich a dataset with structured predictions from an LLM.
#
# The classify_complaint() function below is a simple example of this pattern:
# give Claude a complaint, get back structured JSON with inferred metadata.
#
# PRODUCTION NOTE: If you were classifying thousands of complaints, you would
# want to enable prompt caching by passing the anthropic-beta header:
#   client.messages.create(
#       ...,
#       extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"},
#   )
# and marking the static system prompt with cache_control={"type": "ephemeral"}.
# This tells Anthropic's API to cache the compiled system prompt across calls,
# dramatically reducing cost and latency when the system prompt is reused.
# See: https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching
# =============================================================================

import os
import json
import anthropic

# 1. Initialize the Anthropic client (reads ANTHROPIC_API_KEY from environment)
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


# 2. Define the classification skill
def classify_complaint(client, complaint_text):
    """
    Classify a single customer complaint using Claude.

    Returns a dict with keys:
        category  - product category (one of the 10 known categories)
        sentiment - positive / negative / neutral
        urgency   - low / medium / high
        key_issue - one short phrase summarising the core problem

    Returns None if the API call or JSON parsing fails.
    """
    system_prompt = """You are a customer-service analyst. When given a customer complaint, 
classify it and respond with ONLY a valid JSON object — no prose, no markdown fences.

The JSON must have exactly these four keys:
  "category"  : the product/service category the complaint belongs to. Choose one of:
                 Credit card, Mortgage, Student loan, Auto loan, Personal loan,
                 Checking account, Savings account, Debt collection,
                 Credit reporting, Money transfer
  "sentiment" : the overall tone — one of: positive, negative, neutral
  "urgency"   : how urgently the customer needs help — one of: low, medium, high
  "key_issue" : a single short phrase (5 words or fewer) naming the core problem

Example output:
{"category": "Credit card", "sentiment": "negative", "urgency": "high", "key_issue": "unauthorized charge dispute"}"""

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=150,
            system=system_prompt,
            messages=[
                {
                    "role": "user",
                    "content": f"Classify this complaint:\n\n{complaint_text}",
                }
            ],
        )
        raw_text = response.content[0].text.strip()
        result = json.loads(raw_text)
        return result

    except json.JSONDecodeError as e:
        print(f"  [Warning] Could not parse JSON response: {e}")
        print(f"  Raw response was: {raw_text!r}")
        return None
    except Exception as e:
        print(f"  [Error] API call failed: {e}")
        return None


# 3. Run on a small sample to keep cost low during learning
print("Classifying a sample of 5 complaints via Claude...\n")
sample_df = df.sample(5, random_state=42).reset_index(drop=True)

# 4. Collect results
results = []
for i, row in sample_df.iterrows():
    print(f"  [{i + 1}/5] Classifying: {row['complaint_text'][:80]}...")
    classification = classify_complaint(client, row["complaint_text"])
    if classification is not None:
        record = {"complaint_text": row["complaint_text"]}
        record.update(classification)
        results.append(record)
    else:
        print(f"  Skipping row {i} due to classification failure.")

results_df = pd.DataFrame(results)

# 5. Display results
print("\n--- LLM Classification Results ---")
print(results_df.to_string(index=False))

## Stage 4: LLM Summarization & Synthesis

In [ ]:
# =============================================================================
# LLM SUMMARIZATION — Synthesis of Customer Complaint Data
#
# This section demonstrates the core value of LLM-powered feedback analysis:
# SYNTHESIS. Rather than reading hundreds of individual complaints, we ask
# Claude to combine many data points into actionable insight.
#
# Traditional NLP (TF-IDF, topic modeling) surfaces *what words appear*.
# LLMs can surface *what customers actually mean* and *what to do about it*.
#
# The three outputs below map to a real product/ops workflow:
#   - themes        → informs prioritization and roadmap decisions
#   - urgent_issues → feeds escalation queues and incident response
#   - suggestion    → seeds process improvement proposals
# =============================================================================

import json


def summarize_category_complaints(client, category_name, complaints_list):
    """
    Summarize a list of complaint texts for a single product category using Claude.

    Parameters
    ----------
    client : anthropic.Anthropic
        Initialized Anthropic client.
    category_name : str
        Human-readable product category label (used in the prompt and output).
    complaints_list : list[str]
        Raw complaint texts already filtered to this category.

    Returns
    -------
    dict with keys:
        category        - the category name passed in
        themes          - top 3 recurring themes Claude identified
        urgent_issues   - any issues flagged as needing escalation
        suggestion      - one concrete process-improvement recommendation
        complaint_count - number of complaints that were summarized
    """
    # Join complaints into a numbered list so the model can reference them easily
    numbered_complaints = "\n".join(
        f"{i + 1}. {text}" for i, text in enumerate(complaints_list)
    )

    prompt = f"""You are analyzing customer complaints for the "{category_name}" product category.

Below are {len(complaints_list)} customer complaints:

{numbered_complaints}

Please analyze these complaints and provide a structured response with exactly these three sections:

THEMES: List the top 3 recurring themes or issues customers are experiencing. Be specific and concise (one sentence each).

URGENT_ISSUES: Identify any complaints that suggest urgent problems requiring escalation (e.g., security concerns, legal exposure, severe service failures, repeated system outages). If none, write "None identified."

SUGGESTION: Provide one concrete, actionable process improvement the company could implement to address the most common source of frustration. Be specific — avoid generic advice.

Format your response exactly as shown, using the section headers THEMES:, URGENT_ISSUES:, and SUGGESTION:"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = response.content[0].text

    # --- Parse the structured response into discrete fields ---
    def extract_section(text, header, next_headers):
        """Extract text between `header` and the first of `next_headers`."""
        start_marker = f"{header}:"
        start = text.find(start_marker)
        if start == -1:
            return "Not found in response."
        start += len(start_marker)
        # Find where the next section begins so we don't bleed across sections
        end = len(text)
        for nxt in next_headers:
            pos = text.find(f"{nxt}:", start)
            if pos != -1 and pos < end:
                end = pos
        return text[start:end].strip()

    themes = extract_section(raw_text, "THEMES", ["URGENT_ISSUES", "SUGGESTION"])
    urgent_issues = extract_section(raw_text, "URGENT_ISSUES", ["SUGGESTION"])
    suggestion = extract_section(raw_text, "SUGGESTION", [])

    return {
        "category": category_name,
        "themes": themes,
        "urgent_issues": urgent_issues,
        "suggestion": suggestion,
        "complaint_count": len(complaints_list),
    }


# ---------------------------------------------------------------------------
# Build category groups from the DataFrame
# (reuse if already computed earlier in the notebook, otherwise build here)
# ---------------------------------------------------------------------------
category_groups = df.groupby("product_category")["complaint_text"].apply(list).to_dict()

# Run summarization on two categories to keep API costs low during exploration.
# Swap in other keys from category_groups.keys() to explore additional categories.
target_categories = ["Mobile Banking App", "Customer Service"]

summaries = []
for cat in target_categories:
    if cat not in category_groups:
        print(f"Category '{cat}' not found in data. Available: {list(category_groups.keys())}")
        continue

    complaints = category_groups[cat]
    print(f"Summarizing {len(complaints)} complaints for: {cat} ...")
    result = summarize_category_complaints(client, cat, complaints)
    summaries.append(result)

# ---------------------------------------------------------------------------
# Print results in a readable format
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("LLM COMPLAINT SUMMARIES")
print("=" * 70)

for summary in summaries:
    print(f"\nCategory : {summary['category']}  ({summary['complaint_count']} complaints)")
    print("-" * 70)
    # Pretty-print the full dict; indent=2 keeps it scannable in notebook output
    print(json.dumps(summary, indent=2))
    print()

## Stage 5: LangGraph Agentic Workflow

In [ ]:
# =============================================================================
# LANGGRAPH AGENTIC WORKFLOW
# =============================================================================
#
# LangGraph is a library for building stateful, multi-step AI workflows as
# directed graphs. Before writing code, here are the four core concepts:
#
# 1. STATE
#    A TypedDict that holds all data flowing through the graph. Every node
#    reads from and writes to this shared state object. Think of it as the
#    "memory" of your workflow — each step can see what previous steps did.
#
# 2. NODES
#    Plain Python functions that do one unit of work. Each node receives the
#    current state as input and returns a dict containing only the keys it
#    wants to update. LangGraph merges the returned dict back into the state.
#    Nodes are where you put your logic: DB queries, LLM calls, calculations.
#
# 3. EDGES
#    Connections between nodes that define the order of execution. A normal
#    edge always goes A -> B. A conditional edge inspects the current state
#    and routes to different nodes based on the result — this is what makes
#    the workflow "agentic": it can branch based on what it finds.
#
# 4. CONDITIONAL ROUTING
#    A routing function reads the state and returns a string key. LangGraph
#    maps that key to the next node. This is how you implement "if urgency
#    is high, escalate; otherwise, wrap up" without hardcoding a fixed path.
#
# Execution model: invoke() runs nodes in topological order, passing the
# accumulated state forward until it reaches the special END sentinel.
# =============================================================================

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict, Any
import sqlite3

# -----------------------------------------------------------------------------
# STATE DEFINITION
# Every key here is a "slot" in the shared state bag. Nodes read from it and
# return dicts with the keys they want to update. Keys not returned are left
# unchanged — you never need to pass through data you didn't touch.
# -----------------------------------------------------------------------------

class FeedbackAnalysisState(TypedDict):
    category: str                      # input: product category to analyze
    complaints: List[str]              # filled by fetch_complaints_node
    classifications: List[Dict]        # filled by classify_urgency_node
    escalation_needed: bool            # filled by check_escalation_node
    report: str                        # filled by generate_report_node


# -----------------------------------------------------------------------------
# NODE 1: fetch_complaints_node
# Responsibility: pull raw complaint text from SQLite for the given category.
# This node reads state["category"] and writes state["complaints"].
# -----------------------------------------------------------------------------

def fetch_complaints_node(state: FeedbackAnalysisState) -> Dict[str, Any]:
    """Fetch up to 10 complaint texts for the requested product category."""

    conn = sqlite3.connect("../data/complaints.db")
    cursor = conn.cursor()

    # Parameterized query — always use ? placeholders, never f-strings, to
    # prevent SQL injection even in learning projects (good habit to build).
    cursor.execute(
        "SELECT complaint_text FROM complaints WHERE product_category = ? LIMIT 10",
        (state["category"],)   # second arg must be a tuple, hence the trailing comma
    )

    rows = cursor.fetchall()
    conn.close()

    # Each row is a 1-tuple like ("The app crashed...",), so we unpack with [0]
    complaint_texts = [row[0] for row in rows]

    print(f"[fetch_complaints_node] Found {len(complaint_texts)} complaints for '{state['category']}'")

    # Return only the key(s) this node is responsible for updating.
    # LangGraph merges this dict into the existing state automatically.
    return {"complaints": complaint_texts}


# -----------------------------------------------------------------------------
# NODE 2: classify_urgency_node
# Responsibility: label each complaint as "high" or "normal" urgency.
# In a real workflow this would be a Claude API call; here we use keyword
# matching so the notebook runs fast without burning API credits.
# -----------------------------------------------------------------------------

# Keywords that signal a complaint needs urgent attention
URGENT_KEYWORDS = {"urgent", "immediately", "now", "asap", "problem"}

def classify_urgency_node(state: FeedbackAnalysisState) -> Dict[str, Any]:
    """Classify each complaint text as high or normal urgency."""

    classifications = []

    for text in state["complaints"]:
        text_lower = text.lower()
        is_urgent = any(keyword in text_lower for keyword in URGENT_KEYWORDS)
        urgency_label = "high" if is_urgent else "normal"

        classifications.append({
            "text": text,
            "urgency": urgency_label
        })

    high_count = sum(1 for c in classifications if c["urgency"] == "high")
    print(f"[classify_urgency_node] {high_count}/{len(classifications)} complaints flagged as high urgency")

    return {"classifications": classifications}


# -----------------------------------------------------------------------------
# NODE 3: check_escalation_node
# Responsibility: decide whether this category warrants escalation.
# This is a pure decision node — no I/O, just state transformation.
# In a real graph you might call a manager API or send a Slack alert here.
# -----------------------------------------------------------------------------

ESCALATION_THRESHOLD = 2   # escalate if at least this many high-urgency complaints

def check_escalation_node(state: FeedbackAnalysisState) -> Dict[str, Any]:
    """Set escalation_needed=True if high-urgency complaint count meets threshold."""

    high_urgency_count = sum(
        1 for c in state["classifications"] if c["urgency"] == "high"
    )

    escalation_needed = high_urgency_count >= ESCALATION_THRESHOLD

    print(f"[check_escalation_node] escalation_needed={escalation_needed} "
          f"(threshold={ESCALATION_THRESHOLD}, found={high_urgency_count})")

    return {"escalation_needed": escalation_needed}


# -----------------------------------------------------------------------------
# ROUTING FUNCTION (used with conditional edges)
# A routing function inspects the current state and returns a string.
# LangGraph maps that string to the next node name via a dictionary you
# provide when calling add_conditional_edges().
#
# Even though both branches here lead to generate_report, we wire them
# separately so you can see exactly how to implement a real fork:
#   - swap "generate_report" for "escalate_node" in the mapping below and
#     add an escalate_node that sends alerts before generating the report.
# -----------------------------------------------------------------------------

def route_after_escalation_check(state: FeedbackAnalysisState) -> str:
    """Return a routing key based on whether escalation is needed."""
    if state["escalation_needed"]:
        return "escalate_path"
    else:
        return "normal_path"


# -----------------------------------------------------------------------------
# NODE 4: generate_report_node
# Responsibility: produce the final human-readable summary of the analysis.
# This is the terminal node — after it runs the graph reaches END.
# -----------------------------------------------------------------------------

def generate_report_node(state: FeedbackAnalysisState) -> Dict[str, Any]:
    """Build a plain-text summary report from all collected state data."""

    total_complaints = len(state["complaints"])
    high_urgency_count = sum(
        1 for c in state["classifications"] if c["urgency"] == "high"
    )

    lines = [
        "=" * 50,
        "COMPLAINT ANALYSIS REPORT",
        "=" * 50,
        f"Category        : {state['category']}",
        f"Complaints found: {total_complaints}",
        f"High urgency    : {high_urgency_count}",
        f"Escalation flag : {'YES — requires immediate attention' if state['escalation_needed'] else 'No'}",
        "-" * 50,
    ]

    high_urgency_items = [c for c in state["classifications"] if c["urgency"] == "high"]
    if high_urgency_items:
        lines.append("High-urgency complaints:")
        for i, item in enumerate(high_urgency_items, start=1):
            preview = item["text"][:120] + "..." if len(item["text"]) > 120 else item["text"]
            lines.append(f"  {i}. {preview}")
    else:
        lines.append("No high-urgency complaints detected.")

    lines.append("=" * 50)

    return {"report": "\n".join(lines)}


# -----------------------------------------------------------------------------
# GRAPH ASSEMBLY
# The order of add_node() calls does not determine execution order — edges do.
# Think of add_node() as "registering" a node with a name, and add_edge() as
# drawing arrows on a flowchart.
# -----------------------------------------------------------------------------

workflow = StateGraph(FeedbackAnalysisState)

workflow.add_node("fetch_complaints",  fetch_complaints_node)
workflow.add_node("classify_urgency",  classify_urgency_node)
workflow.add_node("check_escalation",  check_escalation_node)
workflow.add_node("generate_report",   generate_report_node)

workflow.set_entry_point("fetch_complaints")

workflow.add_edge("fetch_complaints", "classify_urgency")
workflow.add_edge("classify_urgency",  "check_escalation")

# Conditional edge: routing function decides which branch to take.
# Both branches currently lead to generate_report — swap "escalate_path"
# to a real "escalate_node" to add alert logic before the final report.
workflow.add_conditional_edges(
    "check_escalation",
    route_after_escalation_check,
    {
        "escalate_path": "generate_report",
        "normal_path":   "generate_report",
    }
)

workflow.add_edge("generate_report", END)

app = workflow.compile()


# -----------------------------------------------------------------------------
# RUN THE GRAPH on two categories
# -----------------------------------------------------------------------------

for category in ["Mobile Banking App", "Customer Service"]:
    print(f"\nRunning analysis for category: '{category}'")
    print("-" * 50)

    result = app.invoke({
        "category":          category,
        "complaints":        [],
        "classifications":   [],
        "escalation_needed": False,
        "report":            "",
    })

    print(result["report"])
    print("---")

## Results & Analysis

In [ ]:
import pandas as pd

# --- Crosstab: product category vs sentiment label ---
print("=== Sentiment Distribution by Product Category ===\n")
if 'sentiment_label' in df.columns and 'product_category' in df.columns:
    crosstab = pd.crosstab(df['product_category'], df['sentiment_label'])
    print(crosstab)
else:
    print("(sentiment_label not yet in df — run classification on full dataset to populate)")

print("\n")

# --- Top 5 most complained-about categories ---
print("Top 5 most complained-about categories:")
print(df['product_category'].value_counts().head(5).to_string())

print("\n")

# --- Resolution rate by category ---
print("Resolution rate by category:")
if 'resolved' in df.columns:
    resolution_rates = (
        df.groupby('product_category')['resolved']
        .mean()
        .sort_values(ascending=False)
        .map(lambda x: f"{x:.1%}")
    )
    print(resolution_rates.to_string())
else:
    print("(resolved column not found in df)")

## What You Built

- **Data loading and chunking** — queried a SQLite database into a pandas DataFrame and split records into manageable batches for LLM processing
- **LLM classification with the Anthropic API** — prompted Claude to label complaint sentiment and category, parsing structured JSON responses back into tabular data
- **LLM synthesis and summarization** — aggregated per-complaint labels into category-level summaries using a second LLM call with a different prompt strategy
- **LangGraph agentic workflow with state, nodes, and conditional routing** — built a stateful multi-step graph where each node performs a discrete task and edges route based on intermediate results
- **Structured reporting** — assembled LLM outputs into a coherent report object, demonstrating how to take raw model responses all the way to a deliverable artifact

## Key Patterns for Your New Role

- **Data chunking strategies** — batch your records to stay within context limits and control cost; chunk size is a tunable parameter, not a fixed constant
- **Prompt design for structured JSON output** — instruct the model explicitly on output format, use a system prompt to set the contract, and always validate the parsed result
- **LangGraph for multi-step AI workflows** — model each logical step as a node, keep state in a typed dict, and use conditional edges to handle branches (e.g., escalation, retries, skips)
- **Cost management** — work on small samples and fast/cheap models (`haiku`) during iteration; swap to a larger model only for final validation or production runs
- **"Skills" and agents as reusable LLM-powered functions** — each node in your graph is effectively a skill; writing them as plain Python functions makes them testable, composable, and easy to swap out

## Next Steps to Explore

- **Add prompt caching** — use Anthropic's cache-control headers on your system prompt to cut costs significantly when running the same prompt structure across many records
- **Build a RAG layer over complaints** — embed complaints with `voyage-3` or a local model, store vectors in a lightweight store (e.g., ChromaDB), and retrieve similar past complaints to ground your prompts
- **Add a human-in-the-loop escalation node in LangGraph** — route low-confidence or high-severity classifications to a human review queue before the report is finalized
- **Connect to a real feedback API** — replace the SQLite seed data with a live pull from a CRM, Zendesk, or survey platform to practice on production-shaped data
- **Add evaluation (evals) for LLM outputs** — build a small labeled ground-truth set and score your classifier's precision/recall; use this to iterate on prompts systematically rather than by intuition